# T0.3 · Title and abstract screening

Prepares `screening.csv` from the T0.2 corpus, summarises the first pass made by AI, lets it be reviewed record by record and measures progress.

- **Input:** `../T0.2-search-and-deduplication/corpus.csv`, the criteria in `../T0.1-protocol/screening-criteria.md` and `screening_ai.csv` (first pass).
- **Output:** `screening.csv`, with `decision` (`include`, `exclude`, `doubt`), `reason` (E1 to E7) and `notes`. It only holds human decisions.
- **To re-run:** run everything. Decisions already saved are kept; new corpus records are added as pending.

In [1]:
import re, textwrap, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
from IPython.display import clear_output

CORPUS = Path("../T0.2-search-and-deduplication/corpus.csv")
CONTROL = Path("../T0.1-protocol/control_set.csv")
SCREENING = Path("screening.csv")
FIRST_PASS = Path("screening_ai.csv")
DECISIONS = ["include", "exclude", "doubt"]
REASONS = {
    "E1": "digital only", "E2": "diagnosis without selection", "E3": "other domain",
    "E4": "test technique or structure without selection", "E5": "not a primary study",
    "E6": "redundant version", "E7": "language",
}
corpus = pd.read_csv(CORPUS, dtype=str).fillna("").set_index("id")
len(corpus)

2632

## 1. Hint and reading order

Each record gets a score from keywords in the title and abstract. It serves two purposes: ordering the reading, from most to least likely, and giving a hint. **It decides nothing**: it is an aid to read faster and it can be wrong.

| Hint | Meaning |
| --- | --- |
| `core` | Mentions analog and accumulates several terms on the selection of points, stimuli or tests, testability or ambiguity groups |
| `no analog` | Mentions nothing analog in the title or abstract (probably E1 or E3). These go last |

In [2]:
ACTION = r"(select|optim|choos|choice|placement|minimi|reduc|compact|generat|order)"
OBJECT = r"(test[- ]?points?|test[- ]?nodes?|measurement (nodes?|points?)|test frequenc\w*|stimul\w*|excitation\w*|test signals?|test sets?|test vectors?|specification tests?)"
CORE = [
    (3, OBJECT + r"\W+(\w+\W+){0,3}" + ACTION),
    (3, ACTION + r"\w*\W+(\w+\W+){0,3}" + OBJECT),
    (3, r"ambiguity (group|set)|test compaction|test selection|adaptive test|test (pattern )?generation|\batpg\b"),
    (2, r"testability|diagnosab|fault dictionar"),
]
ANALOG = r"\banalog|mixed[- ]signal|\bams\b"

def score(title, abstract):
    text = f"{title} {abstract}".lower()
    points = sum(p for p, pattern in CORE if re.search(pattern, text))
    if re.search(ANALOG, title.lower()):
        points += 1
    if not re.search(ANALOG, text):
        hint = "no analog"
    elif points >= 6:
        hint = "core"
    else:
        hint = ""
    return points, hint

sheet = corpus[["year", "title"]].copy()
sheet[["points", "hint"]] = [score(t, a) for t, a in zip(corpus["title"], corpus["abstract"])]
sheet["last"] = sheet["hint"] == "no analog"
sheet = sheet.sort_values(["last", "points", "year"], ascending=[True, False, False], kind="stable")
sheet["order"] = range(1, len(sheet) + 1)

for col in ["decision", "reason", "notes"]:
    sheet[col] = ""
if SCREENING.exists():
    previous = pd.read_csv(SCREENING, dtype=str).fillna("").set_index("id")
    common = sheet.index.intersection(previous.index)
    sheet.loc[common, ["decision", "reason", "notes"]] = previous.loc[common, ["decision", "reason", "notes"]]

COLUMNS = ["order", "hint", "year", "title", "decision", "reason", "notes"]
def save():
    sheet[COLUMNS].to_csv(SCREENING, index_label="id", encoding="utf-8")
save()
sheet["hint"].replace("", "(no hint)").value_counts().rename("records").to_frame()

,records
hint,
(no hint),1655
no analog,710
core,267


## 2. First pass (AI)

`screening_ai.csv` holds a proposed decision for each record, made by an AI assistant with the criteria of the protocol. It is a first approximation and does not replace the human decision.

**Limitation:** the proposal was made by reading the title and only the beginning of the abstract: about 400 characters in the first 300 records of the reading order and less and less afterwards, down to about 110 in the last 230.

In [3]:
ai = pd.read_csv(FIRST_PASS, dtype=str).fillna("").set_index("id")
assert set(ai.index) == set(sheet.index), "screening_ai.csv does not cover the current corpus"

by_decision = ai["decision"].value_counts().reindex(DECISIONS, fill_value=0)
print(by_decision.rename("proposal").to_string(), "\n")
by_reason = ai.loc[ai["decision"] == "exclude", "reason"].value_counts().reindex(list(REASONS), fill_value=0)
pd.DataFrame({"reason": [REASONS[m] for m in by_reason.index], "records": by_reason.values}, index=by_reason.index)

decision
include     468
exclude    1890
doubt       274 



,reason,records
reason,,
E1,digital only,153
E2,diagnosis without selection,336
E3,other domain,572
E4,test technique or structure without selection,622
E5,not a primary study,124
E6,redundant version,0
E7,language,83


The notes of the first pass flag the records closest to the CERTA proposal. Count by topic:

In [4]:
TOPICS = {
    "self-test with a microcontroller or embedded": r"self-test|microcontroller",
    "board level": r"board",
    "risk, escapes, defect level or yield": r"escape|defect|dppm|yield|severity|probab|iso 26262",
    "confidence or statistics of the coverage": r"confidence|statistic|coverage",
    "biomedical": r"ecg|cardiac|bioimpedance",
}
rows = []
for topic, pattern in TOPICS.items():
    sel = ai[ai["notes"].str.lower().str.contains(pattern) & (ai["decision"] != "exclude")]
    rows.append((topic, len(sel), ", ".join(sel.index[:8]) + (" …" if len(sel) > 8 else "")))
pd.DataFrame(rows, columns=["topic", "records (include or doubt)", "examples"])

,topic,records (include or doubt),examples
0,self-test with a microcontroller or embedded,27,"R1825, R1752, R1709, R1451, R0520, R1440, R139..."
1,board level,19,"R1107, R1395, R0105, R0021, R0022, R1403, R082..."
2,"risk, escapes, defect level or yield",23,"R1717, R1190, R0457, R0160, R1835, R1436, R117..."
3,confidence or statistics of the coverage,12,"R1717, R1190, R1010, R0693, R0506, R0507, R020..."
4,biomedical,2,"R1271, R1479"


## 3. Review

Set `SCREEN = True` and run the cell. It shows the pending records one at a time, with the proposal of the first pass, and saves each answer to `screening.csv` straight away, so the work can be left and resumed.

`ONLY` limits the review to the records with a given proposal (`"doubt"`, `"include"` or `"exclude"`); with `None` all of them are visited.

| Key | Effect |
| --- | --- |
| Enter | accept the proposal |
| `i` | include |
| `d` | doubt (goes to full text) |
| `1` to `7` | exclude by E1 to E7 |
| `s` | skip without deciding |
| `u` | undo the previous answer |
| `q` | quit |

A note can be typed after the key, for example `5 review` or `d unclear whether it chooses nodes`.

In [5]:
def show(rid, done, total):
    r, h = corpus.loc[rid], sheet.loc[rid]
    clear_output(wait=True)
    p = ai.loc[rid]
    proposal = " ".join(x for x in (p["decision"], p["reason"], p["notes"] and "· " + p["notes"]) if x)
    print(f"{done}/{total} screened · {rid} · hint: {h['hint'] or '-'}")
    print(f"proposal: {proposal}")
    extras = [x for x in (r["language"], r["possible_version"] and "possible version " + r["possible_version"]) if x]
    print(f"{r['year']} · {r['source'][:90]}" + (" · " + " · ".join(extras) if extras else ""))
    print()
    print(textwrap.fill(r["title"].upper(), 110))
    print()
    print(textwrap.fill(r["abstract"] or "(no abstract)", 110))
    print()
    print("Enter accept · i include · d doubt · 1-7 exclude E1-E7 · s skip · u undo · q quit")

def screen(only=None, read=input):
    pending = [i for i in sheet.index if sheet.at[i, "decision"] == "" and only in (None, ai.at[i, "decision"])]
    total, history, pos = len(sheet), [], 0
    while pos < len(pending):
        rid = pending[pos]
        show(rid, (sheet["decision"] != "").sum(), total)
        key, _, note = read("> ").strip().partition(" ")
        key = key.lower()
        if key == "q":
            break
        if key == "u" and history:
            pos = history.pop()
            sheet.loc[pending[pos], ["decision", "reason", "notes"]] = ""
            save()
            continue
        if key == "s":
            pos += 1
            continue
        if key == "":
            decision, reason = ai.at[rid, "decision"], ai.at[rid, "reason"]
        elif key == "i":
            decision, reason = "include", ""
        elif key == "d":
            decision, reason = "doubt", ""
        elif key in list("1234567"):
            decision, reason = "exclude", "E" + key
        else:
            continue                      # unrecognised key: ask again
        sheet.loc[rid, ["decision", "reason", "notes"]] = [decision, reason, note.strip()]
        save()
        history.append(pos)
        pos += 1
    clear_output(wait=True)
    print((sheet["decision"] != "").sum(), "of", total, "screened")

SCREEN = False
ONLY = "doubt"
if SCREEN:
    screen(ONLY)

## 4. Progress

Checks that the saved human decisions are valid, summarises progress and measures the agreement with the first pass. It can be run at any time.

In [6]:
state = pd.read_csv(SCREENING, dtype=str).fillna("").set_index("id")

errors = state[
    ~state["decision"].isin(DECISIONS + [""])
    | ((state["decision"] == "exclude") & ~state["reason"].isin(REASONS))
    | ((state["decision"] != "exclude") & (state["reason"] != ""))
]
if len(errors):
    print("Rows with an invalid decision or reason:")
    display(errors[["title", "decision", "reason"]])

done = (state["decision"] != "").sum()
print(f"Screened: {done} of {len(state)} ({done / len(state):.0%})")
summary = state["decision"].replace("", "pending").value_counts().reindex(DECISIONS + ["pending"], fill_value=0)
summary.rename("records").to_frame()

Screened: 0 of 2632 (0%)


,records
decision,
include,0
exclude,0
doubt,0
pending,2632


In [7]:
reasons = state.loc[state["decision"] == "exclude", "reason"].value_counts().reindex(list(REASONS), fill_value=0)
pd.DataFrame({"reason": [REASONS[m] for m in reasons.index], "records": reasons.values}, index=reasons.index)

,reason,records
reason,,
E1,digital only,0
E2,diagnosis without selection,0
E3,other domain,0
E4,test technique or structure without selection,0
E5,not a primary study,0
E6,redundant version,0
E7,language,0


Agreement between the human review and the first pass, over the records already reviewed. A doubt counts as an inclusion.

In [8]:
rev = state[state["decision"] != ""].join(ai, rsuffix="_ai")
if len(rev) == 0:
    print("No records reviewed yet.")
else:
    human = rev["decision"] != "exclude"
    machine = rev["decision_ai"] != "exclude"
    agreement = (human == machine).mean()
    chance = human.mean() * machine.mean() + (1 - human.mean()) * (1 - machine.mean())
    kappa = (agreement - chance) / (1 - chance) if chance < 1 else float("nan")
    print(f"Reviewed: {len(rev)} · agreement: {agreement:.0%} · Cohen's kappa: {kappa:.2f}")
    display(pd.crosstab(rev["decision"].rename("human"), rev["decision_ai"].rename("first pass")))

No records reviewed yet.


Check against the control set: none of its works should end up excluded. If one is, that decision or the criterion behind it has to be reviewed.

In [9]:
control = pd.read_csv(CONTROL, dtype=str).fillna("")
by_doi = corpus[corpus["doi"] != ""].reset_index().set_index("doi")["id"].to_dict()
control["corpus_id"] = control["doi"].str.lower().map(by_doi).fillna("")
control["first_pass"] = control["corpus_id"].map(ai["decision"]).fillna("")
control["decision"] = control["corpus_id"].map(state["decision"]).fillna("")
in_corpus = control[control["corpus_id"] != ""]
print("Control works in the corpus:", len(in_corpus))
print("Excluded in the first pass:", (in_corpus["first_pass"] == "exclude").sum())
print("Excluded in the human review:", (in_corpus["decision"] == "exclude").sum())
in_corpus[in_corpus[["first_pass", "decision"]].isin(["exclude", "doubt"]).any(axis=1)][["id", "reference", "corpus_id", "first_pass", "decision"]]

Control works in the corpus: 23
Excluded in the first pass: 0
Excluded in the human review: 0


,id,reference,corpus_id,first_pass,decision


## Conclusions and notes

First pass (AI) over the 2632 records of the corpus:

- **468 include, 274 doubt and 1890 exclude.** 742 records would go to full text if the human review confirms the proposal.
- **Exclusion reasons:** E4, test technique or structure without selection (622); E3, other domain (572); E2, diagnosis without selection (336); E1, digital only (153); E5, not a primary study (124); E7, language (83).
- **Control set:** the 23 works present in the corpus are all proposed as include.
- **Redundant versions:** the 48 groups get the same decision for all their records. Reason E6 has not been applied; it is decided at full text.
- **Tracking records (750):** 32 include, 75 doubt and 643 exclude. Almost everything they contribute comes from the forward citations; the author search adds 384 records and only 4 are proposed as include, because the relevant work of those authors was already in the corpus.
- **System-level test selection:** about 30 tracking records deal with test selection using dependency matrices. They are left as doubt, because the protocol only includes them if they are evaluated on an analog circuit and that cannot be seen in the abstract.
- **Reliability:** the proposal is safer at the beginning of the reading order, where more of the abstract was read, and at the extremes. Even less was read for the tracking records: between 300 and 70 characters of abstract, and only the title for the last 330. The boundaries between E2 and E4, and between include and doubt in test generation and coverage metrics, are the most debatable.
- **Pending:** human review. The most useful order is to start with the 274 doubts (`ONLY = "doubt"`), continue with the inclusions and review a sample of the exclusions.

### Notes